In [39]:
from pyspark.sql import SparkSession, Window , functions as F
import numpy as np
from pyspark.sql.types import StructField, StructType, LongType
from datetime import date

In [2]:
spark = SparkSession.builder.master("local[*]").appName("tsd-batch").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/18 14:49:23 WARN Utils: Your hostname, Sandeep, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/18 14:49:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/18 14:49:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
pwd

'/mnt/c/Users/ranas/OneDrive/Desktop/TSD/notebooks'

In [4]:
fleet = spark.read.parquet("../data/fleet/")

In [5]:
print("partitions:",fleet.rdd.getNumPartitions())

partitions: 5


In [6]:
fleet.show(10)

+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----+--------+---------+
|          timestamp|  userDataReadIops| userDataWriteIops|     readLatencyMs|    writeLatencyMs|        cpuPercent|     memoryPercent|label|    tier|device_id|
+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----+--------+---------+
|2026-01-02 22:17:00|4066.8545508887896|2298.1653847137045|2.2426114992167387|2.4109942318085404|39.362853346045014| 56.25647008640749|    0|internal|  node_04|
|2026-01-04 08:01:00| 5680.139187674713|  3291.02658556083| 2.022521091496823|2.8533152089852543| 51.03399589131322| 69.67376676092641|    0|internal|  node_04|
|2026-01-03 15:03:00| 5855.537408425921|3572.3475980208214|1.8056102587881968|3.1600466806468073| 46.54299321677242| 66.28276796475005|    0|internal|  node_04|
|2026-01-01 08:45:00|5668.10141336

In [7]:
fleet.groupBy("device_id","tier").count().orderBy("device_id").show()

+---------+----------+-----+
|device_id|      tier|count|
+---------+----------+-----+
|  node_00|production|10000|
|  node_01|production|10000|
|  node_02|  internal|10000|
|  node_03|  internal|10000|
|  node_04|  internal|10000|
+---------+----------+-----+



In [8]:
fleet.printSchema()

root
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- userDataReadIops: double (nullable = true)
 |-- userDataWriteIops: double (nullable = true)
 |-- readLatencyMs: double (nullable = true)
 |-- writeLatencyMs: double (nullable = true)
 |-- cpuPercent: double (nullable = true)
 |-- memoryPercent: double (nullable = true)
 |-- label: long (nullable = true)
 |-- tier: string (nullable = true)
 |-- device_id: string (nullable = true)



In [9]:
FEATURE_MAP = {
    "userDataReadIops":  "iops_data_read",
    "userDataWriteIops": "iops_data_write",
    "readLatencyMs":     "read_latency",
    "writeLatencyMs":    "write_latency",
}

In [10]:
w_roll  = Window.partitionBy("device_id").orderBy("timestamp").rowsBetween(-4, 0)
w_order = Window.partitionBy("device_id").orderBy("timestamp") 

In [11]:
def add_features(df):
    cnt = F.count("*").over(w_roll)          # rows in the 5-window, per device
    for col, prefix in FEATURE_MAP.items():
        df = (df
            .withColumn(f"{prefix}_rolling_mean_5m",
                        F.when(cnt < 5, None).otherwise(F.avg(col).over(w_roll)))
            .withColumn(f"{prefix}_rolling_std_5m",
                        F.when(cnt < 5, None).otherwise(F.stddev(col).over(w_roll)))
            .withColumn(f"{prefix}_pct_change_1h",
                        (F.col(col) - F.lag(col, 60).over(w_order)) / F.lag(col, 60).over(w_order))
        )
    return df

In [12]:
feat = add_features(fleet)

In [13]:
feat.filter("device_id = 'node_00'").orderBy("timestamp").select("timestamp","readLatencyMs","read_latency_rolling_mean_5m","read_latency_pct_change_1h").show(10)

+-------------------+------------------+----------------------------+--------------------------+
|          timestamp|     readLatencyMs|read_latency_rolling_mean_5m|read_latency_pct_change_1h|
+-------------------+------------------+----------------------------+--------------------------+
|2026-01-01 00:00:00| 2.756533044269604|                        NULL|                      NULL|
|2026-01-01 00:01:00| 2.348636175658538|                        NULL|                      NULL|
|2026-01-01 00:02:00| 1.138976444789376|                        NULL|                      NULL|
|2026-01-01 00:03:00| 1.759247050828834|                        NULL|                      NULL|
|2026-01-01 00:04:00|  2.41253484087138|          2.0831855112835465|                      NULL|
|2026-01-01 00:05:00|2.1509676543071703|          1.9620724332910595|                      NULL|
|2026-01-01 00:06:00|1.5743048674032614|          1.8072061716400043|                      NULL|
|2026-01-01 00:07:00| 1.731842

In [14]:
spark_pd = (feat.filter("device_id = 'node_00'")
                .orderBy("timestamp")
                .toPandas())

In [15]:
raw_pd = (fleet.filter("device_id = 'node_00'")
               .select("timestamp", *FEATURE_MAP.keys())
               .orderBy("timestamp")
               .toPandas())

In [16]:
for col, prefix in FEATURE_MAP.items():
    raw_pd[f"{prefix}_rolling_mean_5m"] = raw_pd[col].rolling(5).mean()
    raw_pd[f"{prefix}_rolling_std_5m"]  = raw_pd[col].rolling(5).std()
    raw_pd[f"{prefix}_pct_change_1h"]   = raw_pd[col].pct_change(60)

In [17]:
feat_cols = [f"{p}_{s}" for p in FEATURE_MAP.values()
             for s in ("rolling_mean_5m", "rolling_std_5m", "pct_change_1h")]

In [18]:
for c in feat_cols:
    match = np.allclose(spark_pd[c].values, raw_pd[c].values, rtol=1e-6, atol=1e-6, equal_nan=True)
    print(f"{'Match' if match else 'Discrepency'}  {c}")

Match  iops_data_read_rolling_mean_5m
Match  iops_data_read_rolling_std_5m
Match  iops_data_read_pct_change_1h
Match  iops_data_write_rolling_mean_5m
Match  iops_data_write_rolling_std_5m
Match  iops_data_write_pct_change_1h
Match  read_latency_rolling_mean_5m
Match  read_latency_rolling_std_5m
Match  read_latency_pct_change_1h
Match  write_latency_rolling_mean_5m
Match  write_latency_rolling_std_5m
Match  write_latency_pct_change_1h


In [19]:
c = "iops_data_read_rolling_std_5m"
diff = (spark_pd[c] - raw_pd[c]).abs()
print("max abs diff:", diff.max())
print("relative:", (diff.max() / raw_pd[c].abs().mean()))

max abs diff: 7.608040375473024e-08
relative: 4.0710694086447154e-10


In [20]:
with open("../models/xgboost_tsd_model_fixed.onnx", "rb") as f:
    model_bytes = f.read()
bc_model = spark.sparkContext.broadcast(model_bytes)

In [21]:
FEATURE_ORDER = [
    "userDataReadIops", "userDataWriteIops", "readLatencyMs", "writeLatencyMs", "cpuPercent", "memoryPercent",
    "iops_data_read_rolling_mean_5m", "iops_data_read_rolling_std_5m", "iops_data_read_pct_change_1h",
    "iops_data_write_rolling_mean_5m", "iops_data_write_rolling_std_5m", "iops_data_write_pct_change_1h",
    "read_latency_rolling_mean_5m", "read_latency_rolling_std_5m", "read_latency_pct_change_1h",
    "write_latency_rolling_mean_5m", "write_latency_rolling_std_5m", "write_latency_pct_change_1h",
]

In [35]:
_session = None

In [36]:
def score_partition(batches):
    global _session
    import onnxruntime as ort, numpy as np
    if _session is None:                                   # built on the EXECUTOR, once
        _session = ort.InferenceSession(bc_model.value)
    input_name = _session.get_inputs()[0].name
    for pdf in batches:
        X = pdf[FEATURE_ORDER].to_numpy(dtype=np.float32)  # exact training order
        preds = _session.run(None, {input_name: X})[0]
        pdf = pdf.copy()
        pdf["predicted_class"] = preds.reshape(-1)
        yield pdf

In [37]:
scorable = feat.dropna(subset=FEATURE_ORDER)
out_schema = StructType(scorable.schema.fields + [StructField("predicted_class", LongType())])
scored = scorable.mapInPandas(score_partition, schema=out_schema)
scored.select("device_id", "timestamp", "predicted_class").show(10)

+---------+-------------------+---------------+
|device_id|          timestamp|predicted_class|
+---------+-------------------+---------------+
|  node_01|2026-01-01 01:00:00|              1|
|  node_01|2026-01-01 01:01:00|              0|
|  node_01|2026-01-01 01:02:00|              0|
|  node_01|2026-01-01 01:03:00|              0|
|  node_01|2026-01-01 01:04:00|              0|
|  node_01|2026-01-01 01:05:00|              0|
|  node_01|2026-01-01 01:06:00|              0|
|  node_01|2026-01-01 01:07:00|              0|
|  node_01|2026-01-01 01:08:00|              0|
|  node_01|2026-01-01 01:09:00|              0|
+---------+-------------------+---------------+
only showing top 10 rows


2026-07-18 15:01:18.993771969 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
Traceback (most recent call last):                                              
  File "/root/spark-venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 262, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/spark-venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [38]:
scored.groupBy("label", "predicted_class").count().orderBy("label", "predicted_class").show()

2026-07-18 15:06:10.915619840 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
2026-07-18 15:06:10.915619879 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
[Stage 28:>                                                         (0 + 2) / 2]

+-----+---------------+-----+
|label|predicted_class|count|
+-----+---------------+-----+
|    0|              0|44798|
|    0|              1|  299|
|    1|              0|  332|
|    1|              1| 2168|
|    2|              0|    3|
|    2|              1|    1|
|    2|              2|  674|
|    3|              3|  763|
|    4|              4|  662|
+-----+---------------+-----+



In [40]:
scored_out = scored.withColumn("scoring_date", F.lit(str(date.today())))
scored_out.write.mode("overwrite").partitionBy("scoring_date") \
    .parquet("reports/")

2026-07-18 15:08:34.022832517 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
2026-07-18 15:08:34.023094273 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
                                                                                

In [41]:
summary = (scored.groupBy("tier", "predicted_class").count().orderBy("tier", "predicted_class"))
summary.show()
summary.write.mode("overwrite").parquet("reports/summary/")

+----------+---------------+-----+
|      tier|predicted_class|count|
+----------+---------------+-----+
|  internal|              0|26943|
|  internal|              1| 1560|
|  internal|              2|  431|
|  internal|              3|  477|
|  internal|              4|  409|
|production|              0|18190|
|production|              1|  908|
|production|              2|  243|
|production|              3|  286|
|production|              4|  253|
+----------+---------------+-----+

